<div style="
    position: relative;
    width: 100%;
    height: 100%;
    overflow: hidden;
    border-radius: 10px;
">

<img src="../img/AML_Banner2.png" style="
    width: 100%;
    height: 100%;
    object-fit: cover;
">

<div style="
    position: absolute;
    top: 40%;
    left: 50%;
    transform: translate(-50%, -50%);
    color: white;
    font-size: 40px;
    font-weight: 600;
    text-align: center;
    font-family: Arial, sans-serif;
    text-shadow: 0px 3px 12px rgba(0,0,0,0.6);
">
Classification Model</br>Evaluation
</div>

</div>

### Introduction

Evaluating a model goes beyond simply reporting accuracy scores. It helps us determine how robust our model is in terms of whether it generalises well to unseen data, and highlights areas where improvements may be necessary.

We focus on the main techniques for evaluating machine learning models, covering a range of measures and methods to check how well a model performs. This helps us understand whether a model is reliable and accurate enough to be useful in practice. We will discuss the following key techniques:

#### Classification model metrics:
- *Confusion Matrices*:  </br>
Provide detailed visualisation of where a classification model makes correct or incorrect predictions, identifying true positives, false positives, true negatives, and false negatives clearly.

- *Accuracy*: </br>
Measures overall correctness, useful primarily when classes are balanced and all errors have equal importance.

- *Precision*: </br> 
Focuses on the correctness of positive predictions, especially critical when false positives are costly (e.g., fraud detection, medical diagnostics).

- *Recall (Sensitivity)*:  </br>
Captures the model's ability to identify all true positives, essential when false negatives are highly consequential (e.g., cancer detection, security threats).

- *F1 Score*:  </br>
Provides a balanced metric between precision and recall, ideal for evaluating models on imbalanced datasets, such as detecting rare diseases or fraudulent activities.

- *ROC* (Receiver Operating Characteristic): </br> 
Visualises the trade-off between true positive rate and false positive rate at different threshold settings, useful for comparing multiple models and choosing appropriate classification thresholds.

- *AUC* (Area Under the ROC Curve):  </br>
Provides a single numeric measure summarising the overall ability of the model to discriminate between classes; a higher AUC indicates better overall model performance.

## Evaluating machine learning classification models
### Spam, or Ham?

The *SpamBase dataset*, available from the UCI Machine Learning Repository, is commonly used in machine learning experiments focused on detecting spam emails. The goal of this dataset is to classify emails as either spam (unwanted commercial or malicious emails) or ham (legitimate, non-spam emails).

- There are 4601 emails.
- Features (attributes): 57 numeric attributes
- Features represent word frequency (e.g., "free", "credit", "money"), character frequencies (e.g., exclamation marks "!", dollar signs "$"), and other indicators often found in spam emails.
- Target Variable (Class Label):
- is_spam: Binary class label (1 indicates spam, 0 indicates legitimate).

Imagine we're testing a model designed to detect spam emails and it has been trained on some training data. Assume, we have a test dataset containing 200 emails in total.  Of these, 100 are actually *spam* and 100 are actually genuine (*not spam*). The machine learning model makes predictions about which emails are spam, and we record the predicted label.


### Install Python libraries

In [1]:
!pip install pandas nltk matplotlib seaborn scikit-learn tensorflow tensorflow-datasets

### Loading the dataset

In [2]:
import pandas as pd

# URL for the SpamBase dataset from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data"

# Column names from the dataset documentation
# There are 57 feature columns and 1 label column (spam=1, not spam=0)
column_names= [
    # Word frequency features (percentage of words in the email that match)
    'word_freq_make', 'word_freq_address', 'word_freq_all', 'word_freq_3d', 'word_freq_our',
    'word_freq_over', 'word_freq_remove', 'word_freq_internet', 'word_freq_order', 'word_freq_mail',
    'word_freq_receive', 'word_freq_will', 'word_freq_people', 'word_freq_report', 'word_freq_addresses',
    'word_freq_free', 'word_freq_business', 'word_freq_email', 'word_freq_you', 'word_freq_credit',
    'word_freq_your', 'word_freq_font', 'word_freq_000', 'word_freq_money', 'word_freq_hp',
    'word_freq_hpl', 'word_freq_george', 'word_freq_650', 'word_freq_lab', 'word_freq_labs',
    'word_freq_telnet', 'word_freq_857', 'word_freq_data', 'word_freq_415', 'word_freq_85',
    'word_freq_technology', 'word_freq_1999', 'word_freq_parts', 'word_freq_pm', 'word_freq_direct',
    'word_freq_cs', 'word_freq_meeting', 'word_freq_original', 'word_freq_project', 'word_freq_re',
    'word_freq_edu', 'word_freq_table', 'word_freq_conference',

    # Character frequency features (percentage of characters in the email)
    'char_freq_;', 'char_freq_(', 'char_freq_[', 'char_freq_!', 'char_freq_$', 'char_freq_#',

    # Capital run length features
    'capital_run_length_average',
    'capital_run_length_longest',
    'capital_run_length_total',

    # Target class (1 = spam, 0 = not spam)
    'class'
]

# Load dataset into Pandas DataFrame
spam_df = pd.read_csv(url, header=None, names=column_names)

# Show first 5 rows of the dataset
print(spam_df.head())

   word_freq_make  word_freq_address  word_freq_all  word_freq_3d  \
0            0.00               0.64           0.64           0.0   
1            0.21               0.28           0.50           0.0   
2            0.06               0.00           0.71           0.0   
3            0.00               0.00           0.00           0.0   
4            0.00               0.00           0.00           0.0   

   word_freq_our  word_freq_over  word_freq_remove  word_freq_internet  \
0           0.32            0.00              0.00                0.00   
1           0.14            0.28              0.21                0.07   
2           1.23            0.19              0.19                0.12   
3           0.63            0.00              0.31                0.63   
4           0.63            0.00              0.31                0.63   

   word_freq_order  word_freq_mail  ...  char_freq_;  char_freq_(  \
0             0.00            0.00  ...         0.00        0.000   
1 

### Prepare the train and test data (resampling)


In [3]:
from sklearn.model_selection import train_test_split

# Split dataset into features (X) and target labels (y)
X = spam_df.drop(columns=['class'])
y = spam_df['class']

seed = 7

# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

We now scale the data as a preprocessing step as many of the models we will demonstrate will benefit from it. 

We also set up all our models in one go, and store them in a dictionary:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


# Standardise the features (important for models like SVM and KNN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

seed = 42

# Define models for evaluation in a dictionary for handling multiple models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=seed),
    "Support Vector Machine (SVM)": SVC(kernel="linear"),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=seed)
}


Now we have a dictionary storing our models, we can iterate through each one and train it on the training data:

In [ ]:
# Train each model 
for name, model in models.items():
    print(f"Training  Model: {name}")

    # Train the specific model
    model.fit(X_train, y_train)

### Evaluating the models

#### Confusion Matrices

A *confusion matrix* is a straightforward table used to measure how well a classification model is doing. It clearly shows the number of times the model makes correct and incorrect predictions, helping us quickly understand its performance. 

There are several key values in a confusion matrix:

<div style="text-align: center;">
    <img src="../img/example3_confusion_matrix_illustration.PNG" width="50%">
</div>

#### True Positives (TP):
This is the count of Emails the model correctly identified as spam. So you would count every email that the model predicted *as spam* and is *actually spam*. 
In our example, the model correctly labelled 80 out of 100 actual spam emails, which we define as *TP = 80*.

#### False Negatives (FN):
Emails the model missed (predicted incorrectly as genuine emails, even though they were spam). Count every email the model labelled as *not spam* but was *actually spam*. For illustration, assume that out of the 100 spam emails, the model missed *20 emails* to give *FN = 20*.  *Note*: TP + FN = total actual spam emails, 80 + 20 = 100.

#### False Positives (FP):
Emails incorrectly marked as spam when they were actually genuine. Here, we count emails that the model labelled *as spam* but were *actually not spam*. In our model, the model incorrectly flagged *10 genuine emails* as spam giving *FP = 10*.

#### True Negatives (TN):
Emails correctly recognised as genuine (non-spam). We count emails that the model labelled as not spam and were indeed actually not spam. If we have 100 genuine emails in out data, and the model correctly identified *90* as genuine, we have *TN = 90*.  
*Note*: FP + TN = total actual genuine emails, 10 + 90 = 100.

The confusion matrix breaks down predictions into these four categories, arranged into a table:

<div align="center">

|                   | **Predicted**: Positive  | **Predicted**: Negative |
|-------------------|-------------------------|-------------------------|
| **Actual**: Positive | TP = 80                | FN = 20               |
| **Actual**: Negative | FP = 10                | TN = 90               |

</div>

We have included TP, FP, FN, and TN for reference. In a true confusion matrix we would just observe the numeric values. From this confusion matrix, we can quickly assess how well the model performs:

- Overall, the model made 170 correct predictions out of 200 (85% accuracy).
- Of all emails marked spam (90), 80 actually were spam, meaning precision is calculated as $\frac{80}{90} = 89$%. This measures reliability.
- Of all actual spam emails (100), the model correctly identified 80, meaning recall is $\frac{80}{100} = 80$%. This measures how effectively the model catches real spam.

This breakdown helps us easily spot exactly where the model is getting things right or wrong, and guides us in improving its predictions, whether it's reducing false alerts (false positives) or catching more missed cases (false negatives).

In practical terms, confusion matrices help spot if your spam detection model is missing too many real spam emails (false negatives), or incorrectly marking legitimate emails as spam (false positives):

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

for name, model in models.items():
    # Make predictions
    y_pred = model.predict(X_test)

    # Compute and display confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(5, 4))
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Not Spam", "Spam"], yticklabels=["Not Spam", "Spam"])
    
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    
    plt.title(f"{name}")


### Accuracy
Accuracy is a fundamental metric used to evaluate classification models, measuring the proportion of correctly predicted instances out of the total number of predictions. It is calculated as the ratio of correct predictions to the total dataset size, providing a straightforward way to assess overall model performance. For example, if a model correctly classifies 90 out of 100 test samples, its accuracy is 90%. 

This metric is particularly useful when dealing with balanced datasets, where all classes have roughly equal representation.

However, accuracy can be misleading when applied to imbalanced datasets. 

In cases where one class significantly outnumbers another, a model could achieve high accuracy by simply predicting the majority class while failing to correctly classify the minority class. 

For example, in a fraud detection scenario where 98% of transactions are legitimate, a model that predicts "not fraud" for every instance would still achieve 98% accuracy, despite failing to detect any fraudulent transactions. 

Additionally, accuracy does not distinguish between types of errors. False positives and false negatives are weighted equally, which may not be suitable for applications where certain types of errors carry greater consequences, such as medical diagnoses or security systems. For such cases, precision, recall, and the F1-score provide a more meaningful assessment of model performance.

In [ ]:
from sklearn.metrics import accuracy_score

for name, model in models.items():
    # Make predictions
    y_pred = model.predict(X_test)

    # Compute accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"accuracy: {accuracy:.4f} - {name}")

### Precision, Recall, and F1-score

Precision measures how trustworthy a model's positive predictions are. It answers the question: "of all the instances the model labelled as positive, how many were actually positive?" This becomes particularly important in situations where false positives are costly, such as when legitimate emails are incorrectly flagged as spam or when an innocent transaction is marked as fraudulent. 

Recall, on the other hand, focuses on completeness. It asks: "of all the actual positive cases, how many did the model successfully identify?" This is important in scenarios where missing a positive case carries serious consequences, such as failing to detect a disease in a medical screening task. 

Because there is often a trade-off between precision and recall, the F1-score provides a useful way to balance the two. It combines both metrics into a single value, giving a more holistic view of model performance. 

This is especially valuable when dealing with imbalanced datasets, where one class is much more frequent than the other, and relying on a single metric like accuracy could be misleading.

To calculate these measures, we can import `classification_report` from `sklearn.metrics` and pass in our test data, and the predictions from the model:

In [ ]:
from sklearn.metrics import classification_report

for name, model in models.items():
    # Make predictions
    y_pred = model.predict(X_test)

    # Display classification report
    print(f"classification report - {name}:")
    
    print(classification_report(y_test, y_pred))


In the output above, *support*, *macro avg*, and *weighted avg* are commonly shown in classification reports, especially for multi-class problems, and they help you interpret precision, recall, and F1-score across all classes.

Support refers to how many true instances there are for each class in the dataset. For example, if you are classifying emails and there are 100 spam emails and 900 non-spam emails, the support for the spam class is 100 and for non-spam is 900. This tells you how balanced or imbalanced your dataset is.

Macro average calculates the metric (precision, recall, or F1-score) independently for each class and then takes the simple average across all classes. Every class is treated equally, regardless of how many examples it has. This means macro average is useful when you want to understand how well the model performs across all classes without being biased toward the majority class.

Weighted average, on the other hand, also averages the metric across classes but takes support into account. Each class contributes proportionally based on how many instances it has. This means classes with more data have a bigger influence on the final score. Weighted averages are useful when you want an overall performance measure that reflects the actual class distribution in your dataset.

In short, support tells you how much data each class has, macro average treats all classes equally, and weighted average reflects the dataset imbalance by giving more importance to larger classes.


### ROC Curve and AUC
The *ROC* (Receiver Operating Characteristic) curve shows how well the model balances sensitivity (how well it catches the correct positive cases) and specificity (how well it avoids false alarms). It does this by plotting the true positive rate against the false positive rate at different thresholds.

The *AUC* (Area Under the Curve) gives a single number that summarises how well the model can tell the difference between classes. A higher AUC means better performance:

- 1.0 = perfect separation between classes

- 0.5 = no better than random guessing

ROC curves are useful when you want to see which classes your model finds harder to predict. This is important in imbalanced datasets, where some classes have many more examples than others, meaning accuracy can be misleading:

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))  # Set figure size for plot

# Loop through each model in our dictionary
for name, model in models.items():
    # Get predicted probabilities or decision scores for the positive class (check if the model has this attribute)
    if hasattr(model, "predict_proba"):
        y_probs = model.predict_proba(X_test)[:, 1]  # Use probability for class 1 (e.g. spam)
    else:
        y_probs = model.decision_function(X_test)  # Use decision scores (e.g. for SVM)

    # Calculate false positive rate and true positive rate (the third output "_" is a list of threshold values used to compute fpr and tpr which we ignore).
    fpr, tpr, _ = roc_curve(y_test, y_probs)
    
    roc_auc = auc(fpr, tpr)  # Area Under the Curve (AUC)

    # Plot ROC curve for the current model
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")

# Add diagonal line representing random guessing
plt.plot([0, 1], [0, 1], "k--", label="Random guessing")

# Add axis labels and title
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves for different models")

# Show legend and grid
plt.legend()
plt.grid(True)

### What have we learnt?

For classification tasks, accuracy works best when your classes are balanced and errors carry equal importance. However, in scenarios where classes are imbalanced or the cost of mistakes differs significantly, such as detecting fraud or diagnosing diseases, it's preferable to rely on precision, recall, or the F1 score to evaluate your model. 

Always examine the confusion matrix as well, since it provides clear insight into exactly how your model is making correct and incorrect predictions.

Neural networks often require specialised evaluation methods. We commonly look at accuracy, loss curves, and cross-entropy loss:

- Accuracy can be used as before to indicate overall prediction correctness, but can be misleading if classes are imbalanced. 

- Loss Curves track model improvement over training epochs. When we plot these curves, we should ideally, we should see training and validation losses consistently decrease and eventually stabilise.

- Cross-Entropy Loss specifically measures the divergence between predicted probabilities and actual values, commonly used for classification problems.

In the next part, we will explore these methods and their use for evaluating neural network models. So far, we have looked at binary classification (Spam vs Ham), so let's now consider a multi-class classification problem at the same time.
